# 🛡️ Bi-Spectral Latent Watermarking (BiSLW) Walkthrough

This notebook demonstrates the step-by-step pipeline of **Bi-Spectral Latent Watermarking (BiSLW)**, a robust watermarking scheme for deep generative models. The watermark is injected into the latent frequency bands of Stable Diffusion v1.5's VAE.

### Pipeline Stages:
1. Load SD VAE and trained BiSLW checkpoints.
2. Load and preprocess an image.
3. Encode the image into the latent space.
4. Decompose latents into low and high-frequency bands.
5. Generate a binary watermark payload key.
6. Inject the watermark into frequency sub-bands.
7. Recombine sub-bands and decode back to image space.
8. Measure quality (PSNR & SSIM) and visualize changes.
9. Simulate attacks (JPEG, blur, noise).
10. Extract the watermark and compute bit verification accuracy.

## 1. Environment Setup & Imports

In [ ]:
import os
import sys
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Add project root to path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from latent_watermarking.models.latent_split import LatentSplitter
from latent_watermarking.models.recombination import LatentRecombiner
from latent_watermarking.models.vae_wrapper import VAEWrapper
from latent_watermarking.models.watermark_decoder import WatermarkDecoder
from latent_watermarking.models.watermark_encoder import WatermarkEncoder
from latent_watermarking.tools.evaluation.metrics import PSNR, SSIM

## 2. Model Loading
We initialize the VAE wrapper and load our trained BiSLW sub-band encoders/decoders.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 
                      'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

# Load VAE
vae = VAEWrapper().to(device)

# Load best checkpoint
ckpt_path = '../../models/BestModel/best.pt'
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
w_dim = ckpt.get('config', {}).get('w_dim', 32)
alpha_l = ckpt.get('alpha_l', 0.02)
alpha_h = ckpt.get('alpha_h', 0.01)

splitter = LatentSplitter(mode='dct').to(device)
recombiner = LatentRecombiner(mode='dct').to(device)
encoder_l = WatermarkEncoder(watermark_dim=w_dim).to(device)
encoder_h = WatermarkEncoder(watermark_dim=w_dim).to(device)
decoder_l = WatermarkDecoder(watermark_dim=w_dim).to(device)
decoder_h = WatermarkDecoder(watermark_dim=w_dim).to(device)

encoder_l.load_state_dict(ckpt['encoder_l'])
encoder_h.load_state_dict(ckpt['encoder_h'])
decoder_l.load_state_dict(ckpt['decoder_l'])
decoder_h.load_state_dict(ckpt['decoder_h'])

for m in [encoder_l, encoder_h, decoder_l, decoder_h]:
    m.eval()
print("Models loaded successfully!")

## 3. Image Preprocessing

In [ ]:
def load_image(path, size=256):
    img = Image.open(path).convert('RGB')
    w, h = img.size
    min_dim = min(w, h)
    left = (w - min_dim) // 2
    top = (h - min_dim) // 2
    img = img.crop((left, top, left + min_dim, top + min_dim))
    img = img.resize((size, size), Image.LANCZOS)
    img_np = np.array(img).astype(np.float32) / 255.0
    tensor = torch.from_numpy(img_np).permute(2, 0, 1).unsqueeze(0)
    return tensor * 2 - 1  # Normalize to [-1, 1]

def to_pil(tensor):
    img_np = ((tensor.squeeze(0).permute(1, 2, 0).detach().cpu().numpy() + 1) / 2 * 255).clip(0, 255).astype(np.uint8)
    return Image.fromarray(img_np)

# Load a sample image
img_path = '../demo/examples/portrait.jpg'
img_tensor = load_image(img_path).to(device)
plt.imshow((img_tensor.squeeze(0).permute(1, 2, 0).cpu().numpy() + 1) / 2)
plt.axis('off')
plt.title("Source Image")
plt.show()

## 4. Latent Space Projection & Band Splitting
We encode the image into VAE space, then apply the row-column DCT-II splitter to extract high and low frequency coordinates.

In [ ]:
with torch.no_grad():
    z = vae.encode(img_tensor)
z_l, z_h = splitter(z)

print(f"Latent space shape: {z.shape}")
print(f"Low-frequency sub-band shape: {z_l.shape}")
print(f"High-frequency sub-band shape: {z_h.shape}")

## 5. Generate and Embed Watermark
We generate a pseudo-random watermark sequence and embed it into both frequency sub-bands.

In [ ]:
# Watermark generation
w_true = torch.randn(1, w_dim, device=device)
w_true = (w_true > 0).float() * 2 - 1

# Embed watermark keys
with torch.no_grad():
    z_l_wm = encoder_l(z_l, w_true, alpha=alpha_l)
    z_h_wm = encoder_h(z_h, w_true, alpha=alpha_h)
    z_wm = recombiner(z_l_wm, z_h_wm)
    
    # Reconstruct back to pixel space
    img_wm = vae.decode(z_wm)

print("Watermark embedded. Reconstructed watermarked image successfully.")

## 6. Measure Quality & Visualize Perturbation

In [ ]:
psnr_calc = PSNR(data_range=2.0)
ssim_calc = SSIM(data_range=2.0).to(device)

psnr_val = psnr_calc(img_tensor, img_wm).item()
ssim_val = ssim_calc(img_tensor, img_wm).item()
print(f"PSNR: {psnr_val:.2f} dB")
print(f"SSIM: {ssim_val:.4f}")

# Compute and plot absolute perturbation difference heatmap
diff = np.abs(np.array(to_pil(img_tensor)).astype(float) - np.array(to_pil(img_wm)).astype(float))
diff_enhanced = np.clip(diff * 15, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(to_pil(img_tensor))
axes[0].set_title("Original Image")
axes[0].axis('off')

axes[1].imshow(to_pil(img_wm))
axes[1].set_title("Watermarked Image")
axes[1].axis('off')

im = axes[2].imshow(np.mean(diff_enhanced, axis=2), cmap='hot')
axes[2].set_title("Perturbation (Enhanced 15x)")
axes[2].axis('off')
fig.colorbar(im, ax=axes[2], orientation='vertical', shrink=0.7)

plt.tight_layout()
plt.show()

## 7. Simulate Attacks
We test model resilience by applying simulated distortion (JPEG-70 compression, blurring, noise, resizing).

In [ ]:
def jpeg_sim(images, quality=70):
    scale = max(0.3, quality / 100)
    B, C, H, W = images.shape
    down = F.interpolate(images, size=(int(H*scale), int(W*scale)), mode='bilinear', align_corners=False)
    return F.interpolate(down, size=(H, W), mode='bilinear', align_corners=False)

img_att = jpeg_sim(img_wm, 70)
plt.imshow(to_pil(img_att))
plt.axis('off')
plt.title("JPEG-70 Attacked Image")
plt.show()

## 8. Decode Watermark and Compute Bit Accuracy

In [ ]:
with torch.no_grad():
    z_att = vae.encode(img_att)
    z_att_l, z_att_h = splitter(z_att)
    w_pred = (decoder_l(z_att_l) + decoder_h(z_att_h)) / 2

bits_true = (w_true > 0).float()
bits_pred = (w_pred > 0).float()
bit_acc = (bits_true == bits_pred).float().mean().item()

print(f"Extraction Bit-level Accuracy: {bit_acc * 100:.1f}%")

# Plot confidence comparison bar chart
plt.figure(figsize=(10, 4))
plt.bar(range(w_dim), w_pred[0].cpu().numpy(), color=['green' if t==p else 'red' for t, p in zip(bits_true[0].tolist(), bits_pred[0].tolist())])
plt.axhline(0, color='black', linestyle='--')
plt.title(f"Extracted Bit Confidences (Accuracy: {bit_acc * 100:.1f}%)")
plt.xlabel("Bit index")
plt.ylabel("Extraction logit value")
plt.show()